# Extração de North e Easth, com OCR

Autor:  Viviane da Rosa Sommer

Atualizado: 13/06/2025

## Objetivo: A partir dos vídeos de uma operação, extrair os valores de North e Easth da overlay.

## Importações necessárias

In [ ]:
!pip install easyocr
import cv2
import easyocr
import re
import os
import csv
import tqdm

## Funções necessárias

In [ ]:
def extract_data(frame):
    x, y, w, h = 706, 14, 123, 67
    roi = frame[y:y+h, x:x+w]
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY)
    binary = cv2.GaussianBlur(binary, (5, 5), 0)
    reader = easyocr.Reader(['en'])
    result = reader.readtext(binary)
    north, east = None, None
    if len(result) >= 2:
        match_north = re.search(r"-?\d+", result[0][1])
        match_east = re.search(r"-?\d+", result[1][1])
        north = int(match_north.group()) if match_north else None
        east = int(match_east.group()) if match_east else None
    return north, east

def fix_discrepancies(data):
    for i in range(1, len(data)):
        for key in ["North", "East"]:
            current = int(data[i][key])
            previous = int(data[i - 1][key])
            if i < len(data) - 1:
                next_val = int(data[i + 1][key])
                if abs(current - previous) > 10 and abs(current - next_val) > 10:
                    data[i][key] = int((previous + next_val) / 2)
            else:
                if abs(current - previous) > 10:
                    data[i][key] = previous
    return data

def process_video(video_path):
    video_name = os.path.basename(video_path)
    csv_name = os.path.splitext(video_name)[0] + ".csv"
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video: {video_path}")
        return []
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_interval = int(fps * 5)
    video_data = []
    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_count % frame_interval == 0:
            total_seconds = int(frame_count / fps)
            minutes, seconds = divmod(total_seconds, 60)
            timestamp = f"{minutes}:{seconds:02d}"
            north, east = extract_data(frame)
            if north is not None and east is not None:
                video_data.append({
                    "Video": video_name,
                    "Timestamp": timestamp,
                    "North": north,
                    "East": east
                })
        frame_count += 1
    cap.release()
    video_data = fix_discrepancies(video_data)
    with open(csv_name, mode="w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=["Video", "Timestamp", "North", "East"])
        writer.writeheader()
        writer.writerows(video_data)
    print(f"Video data '{video_name}' saved in '{csv_name}'")
    return video_data

In [ ]:
videos = [
    '../Videos-Teste/downloads/SUB8AG24-108-OS006000685892-2024-06-03-T-13-09-47-D030.mp4',
    '../Videos-Teste/downloads/SUB8AG24-108-OS006000685892-2024-06-03-T-13-39-47-D030.mp4',
    '../Videos-Teste/downloads/SUB8AG24-108-OS006000685892-2024-06-03-T-14-09-43-D030.mp4',
    '../Videos-Teste/downloads/SUB8AG24-108-OS006000685892-2024-06-03-T-14-39-41-D030.mp4',
    '../Videos-Teste/downloads/SUB8AG24-108-OS006000685892-2024-06-03-T-15-09-39-D030.mp4',
    '../Videos-Teste/downloads/SUB8AG24-108-OS006000685892-2024-06-03-T-15-39-37-D030.mp4',
    '../Videos-Teste/downloads/SUB8AG24-108-OS006000685892-2024-06-03-T-16-09-35-D030.mp4',
    '../Videos-Teste/downloads/SUB8AG24-108-OS006000685892-2024-06-03-T-16-39-33-D030.mp4',
    '../Videos-Teste/downloads/SUB8AG24-108-OS006000685892-2024-06-03-T-17-09-31-D030.mp4',
    '../Videos-Teste/downloads/SUB8AG24-108-OS006000685892-2024-06-03-T-17-39-29-D030.mp4',
    '../Videos-Teste/downloads/SUB8AG24-108-OS006000685892-2024-06-03-T-18-09-25-D030.mp4'
]

for video in tqdm(videos, desc="Processando vídeos"):
    process_video(video)

print("Processamento concluído.")